# Phase 2 Benchmark: TPC-H Q1 + Q6

> Closeout benchmark for Phase 2 on active MXFrame lazy APIs.

This notebook benchmarks:
- MXFrame forced CPU
- MXFrame forced GPU (when MAX-compatible GPU is available)
- Polars baseline (optional)
- Pandas baseline

Queries covered:
- **Q1** grouped aggregates by `l_returnflag`, `l_linestatus`
- **Q6** filtered global revenue aggregate

In [ ]:
import time
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc

from mxframe import LazyFrame, Scan, col, lit

try:
    import polars as pl
    POLARS_AVAILABLE = True
except ImportError:
    POLARS_AVAILABLE = False
    print("Polars not installed: skipping Polars baseline.")

from max import driver as _driver

## 1) Synthetic lineitem data

In [ ]:
CUTOFF_Q1 = 10471          # 1998-09-02
Q6_DATE_LO = 8766          # 1994-01-01
Q6_DATE_HI = 9131          # 1995-01-01 (exclusive)
Q6_DISC_LO = 0.05
Q6_DISC_HI = 0.07
Q6_QTY_HI = 24.0

def make_lineitem_arrow(n: int = 1_000_000, seed: int = 42) -> pa.Table:
    rng = np.random.default_rng(seed)

    rf_choices = rng.integers(0, 3, size=n)
    ls_choices = rng.integers(0, 2, size=n)

    rf = np.array(["A", "N", "R"], dtype=object)[rf_choices]
    ls = np.array(["F", "O"], dtype=object)[ls_choices]

    quantity = rng.uniform(1.0, 50.0, size=n).astype(np.float32)
    extendedprice = rng.uniform(900.0, 100_000.0, size=n).astype(np.float32)
    discount = rng.uniform(0.0, 0.10, size=n).astype(np.float32)
    tax = rng.uniform(0.0, 0.08, size=n).astype(np.float32)
    shipdate = rng.integers(8_000, 10_550, size=n).astype(np.int32)

    return pa.table({
        "l_returnflag": rf.tolist(),
        "l_linestatus": ls.tolist(),
        "l_quantity": quantity,
        "l_extendedprice": extendedprice,
        "l_discount": discount,
        "l_tax": tax,
        "l_shipdate": shipdate,
    })

N = 1_000_000
lineitem = make_lineitem_arrow(N)
print(f"Generated {lineitem.num_rows:,} rows with columns: {lineitem.column_names}")

Generated 1,000,000 rows with columns: ['l_returnflag', 'l_linestatus', 'l_quantity', 'l_extendedprice', 'l_discount', 'l_tax', 'l_shipdate']


## 2) Query implementations (MXFrame, Polars, Pandas)

In [ ]:
def _lf(tbl: pa.Table) -> LazyFrame:
    return LazyFrame(Scan(tbl))

def _sort_q1_arrow(tbl: pa.Table) -> pa.Table:
    idx = pc.sort_indices(tbl, sort_keys=[("l_returnflag", "ascending"), ("l_linestatus", "ascending")])
    return tbl.take(idx)

def run_q1_mxframe(tbl: pa.Table, device: str = "cpu") -> pa.Table:
    lf = _lf(tbl)
    out = (
        lf
        .filter(col("l_shipdate") <= lit(CUTOFF_Q1))
        .groupby("l_returnflag", "l_linestatus")
        .agg(
            col("l_quantity").sum().alias("sum_qty"),
            col("l_extendedprice").sum().alias("sum_base_price"),
            (col("l_extendedprice") * (lit(1.0) - col("l_discount"))).sum().alias("sum_disc_price"),
            (col("l_extendedprice") * (lit(1.0) - col("l_discount")) * (lit(1.0) + col("l_tax"))).sum().alias("sum_charge"),
            col("l_quantity").mean().alias("avg_qty"),
            col("l_extendedprice").mean().alias("avg_price"),
            col("l_discount").mean().alias("avg_disc"),
            col("l_quantity").count().alias("count_order"),
        )
        .compute(device=device)
    )
    return _sort_q1_arrow(out)

def run_q6_mxframe(tbl: pa.Table, device: str = "cpu") -> pa.Table:
    lf = _lf(tbl)
    return (
        lf
        .filter(
            (col("l_shipdate") >= lit(Q6_DATE_LO))
            & (col("l_shipdate") < lit(Q6_DATE_HI))
            & (col("l_discount") >= lit(Q6_DISC_LO))
            & (col("l_discount") <= lit(Q6_DISC_HI))
            & (col("l_quantity") < lit(Q6_QTY_HI))
        )
        .groupby()
        .agg((col("l_extendedprice") * col("l_discount")).sum().alias("revenue"))
        .compute(device=device)
    )

def run_q1_pandas(tbl: pa.Table) -> pd.DataFrame:
    pdf = tbl.to_pandas()
    q1 = pdf[pdf["l_shipdate"] <= CUTOFF_Q1].copy()
    q1["disc_price"] = q1["l_extendedprice"] * (1.0 - q1["l_discount"])
    q1["charge"] = q1["disc_price"] * (1.0 + q1["l_tax"])

    out = (
        q1.groupby(["l_returnflag", "l_linestatus"], as_index=False)
        .agg(
            sum_qty=("l_quantity", "sum"),
            sum_base_price=("l_extendedprice", "sum"),
            sum_disc_price=("disc_price", "sum"),
            sum_charge=("charge", "sum"),
            avg_qty=("l_quantity", "mean"),
            avg_price=("l_extendedprice", "mean"),
            avg_disc=("l_discount", "mean"),
            count_order=("l_quantity", "count"),
        )
        .sort_values(["l_returnflag", "l_linestatus"])
        .reset_index(drop=True)
    )
    return out

def run_q6_pandas(tbl: pa.Table) -> float:
    pdf = tbl.to_pandas()
    mask = (
        (pdf["l_shipdate"] >= Q6_DATE_LO)
        & (pdf["l_shipdate"] < Q6_DATE_HI)
        & (pdf["l_discount"] >= Q6_DISC_LO)
        & (pdf["l_discount"] <= Q6_DISC_HI)
        & (pdf["l_quantity"] < Q6_QTY_HI)
    )
    return float((pdf.loc[mask, "l_extendedprice"] * pdf.loc[mask, "l_discount"]).sum())

def run_q1_polars(tbl: pa.Table):
    if not POLARS_AVAILABLE:
        return None
    pl_df = pl.from_arrow(tbl)
    return (
        pl_df
        .filter(pl.col("l_shipdate") <= CUTOFF_Q1)
        .with_columns([
            (pl.col("l_extendedprice") * (1.0 - pl.col("l_discount"))).alias("disc_price"),
            (pl.col("l_extendedprice") * (1.0 - pl.col("l_discount")) * (1.0 + pl.col("l_tax"))).alias("charge"),
        ])
        .group_by(["l_returnflag", "l_linestatus"])
        .agg([
            pl.col("l_quantity").sum().alias("sum_qty"),
            pl.col("l_extendedprice").sum().alias("sum_base_price"),
            pl.col("disc_price").sum().alias("sum_disc_price"),
            pl.col("charge").sum().alias("sum_charge"),
            pl.col("l_quantity").mean().alias("avg_qty"),
            pl.col("l_extendedprice").mean().alias("avg_price"),
            pl.col("l_discount").mean().alias("avg_disc"),
            pl.col("l_quantity").count().alias("count_order"),
        ])
        .sort(["l_returnflag", "l_linestatus"])
)

def run_q6_polars(tbl: pa.Table):
    if not POLARS_AVAILABLE:
        return None
    pl_df = pl.from_arrow(tbl)
    out = (
        pl_df
        .filter(
            (pl.col("l_shipdate") >= Q6_DATE_LO)
            & (pl.col("l_shipdate") < Q6_DATE_HI)
            & (pl.col("l_discount") >= Q6_DISC_LO)
            & (pl.col("l_discount") <= Q6_DISC_HI)
            & (pl.col("l_quantity") < Q6_QTY_HI)
        )
        .select((pl.col("l_extendedprice") * pl.col("l_discount")).sum().alias("revenue"))
    )
    return float(out["revenue"][0])

q1_cpu_preview = run_q1_mxframe(lineitem, device="cpu")
q6_cpu_preview = run_q6_mxframe(lineitem, device="cpu")
print("MXFrame Q1 CPU preview:")
print(q1_cpu_preview.to_pandas().to_string(index=False))
print(f"MXFrame Q6 CPU preview revenue: {float(q6_cpu_preview.column('revenue')[0].as_py()):.4f}")

MXFrame Q1 CPU preview:
l_returnflag l_linestatus    sum_qty  sum_base_price  sum_disc_price   sum_charge   avg_qty    avg_price  avg_disc  count_order
           A            F 4103107.75    8131914752.0    7725578752.0 8034785792.0 25.447050 50433.292969  0.049952     161241.0
           A            O 4139794.75    8178190848.0    7769572352.0 8080540672.0 25.557285 50488.582031  0.050003     161981.0
           N            F 4119782.00    8149893120.0    7741892096.0 8051399168.0 25.460300 50366.433594  0.049968     161812.0
           N            O 4113073.75    8155671552.0    7748141568.0 8058702336.0 25.469685 50502.953125  0.049982     161489.0
           R            F 4121115.75    8132211200.0    7725741568.0 8035167232.0 25.521063 50360.796875  0.050007     161479.0
           R            O 4101858.50    8113100800.0    7707712000.0 8016372224.0 25.494167 50425.128906  0.049970     160894.0
MXFrame Q6 CPU preview revenue: 40689616.0000


## 3) Baseline previews

In [ ]:
q1_pd = run_q1_pandas(lineitem)
q6_pd = run_q6_pandas(lineitem)
print("Pandas Q1 preview:")
print(q1_pd.to_string(index=False))
print(f"Pandas Q6 revenue: {q6_pd:.4f}")

if POLARS_AVAILABLE:
    q1_pl = run_q1_polars(lineitem)
    q6_pl = run_q6_polars(lineitem)
    print("Polars Q1 preview:")
    print(q1_pl)
    print(f"Polars Q6 revenue: {q6_pl:.4f}")
else:
    q1_pl = None
    q6_pl = None

Pandas Q1 preview:
l_returnflag l_linestatus    sum_qty  sum_base_price  sum_disc_price   sum_charge   avg_qty    avg_price  avg_disc  count_order
           A            F 4103113.50    8131883520.0    7725592576.0 8034798592.0 25.447086 50433.101562  0.049952       161241
           A            O 4139794.00    8178198016.0    7769561600.0 8080580608.0 25.557281 50488.625000  0.050003       161981
           N            F 4119764.75    8149879296.0    7741915648.0 8051404800.0 25.460194 50366.347656  0.049968       161812
           N            O 4113074.75    8155675136.0    7748204032.0 8058711040.0 25.469690 50502.976562  0.049982       161489
           R            F 4121102.50    8132192256.0    7725764096.0 8035144704.0 25.520981 50360.679688  0.050007       161479
           R            O 4101842.50    8113126400.0    7707748864.0 8016333824.0 25.494068 50425.289062  0.049970       160894
Pandas Q6 revenue: 40689612.0000
Polars Q1 preview:
shape: (6, 10)
┌───────────┬─────

## 4) Correctness checks (Q1 + Q6)

In [ ]:
def _assert_q1_close(mx_tbl: pa.Table, ref_df: pd.DataFrame, label: str, rtol: float = 1e-2, atol: float = 1e-4):
    mx_df = mx_tbl.to_pandas().sort_values(["l_returnflag", "l_linestatus"]).reset_index(drop=True)
    ref_df = ref_df.sort_values(["l_returnflag", "l_linestatus"]).reset_index(drop=True)

    assert len(mx_df) == len(ref_df), f"{label}: row-count mismatch {len(mx_df)} vs {len(ref_df)}"
    for key in ["l_returnflag", "l_linestatus"]:
        assert (mx_df[key].values == ref_df[key].values).all(), f"{label}: key mismatch in {key}"

    value_cols = [
        "sum_qty", "sum_base_price", "sum_disc_price", "sum_charge",
        "avg_qty", "avg_price", "avg_disc", "count_order",
    ]
    for colname in value_cols:
        mx_vals = mx_df[colname].astype(float).to_numpy()
        ref_vals = ref_df[colname].astype(float).to_numpy()
        if not np.allclose(mx_vals, ref_vals, rtol=rtol, atol=atol):
            raise AssertionError(f"{label}: mismatch in {colname}")

def _extract_q6_revenue(mx_tbl: pa.Table) -> float:
    return float(mx_tbl.column("revenue")[0].as_py())

_assert_q1_close(q1_cpu_preview, q1_pd, "Q1 MXFrame CPU vs Pandas")
q6_mx_cpu_val = _extract_q6_revenue(q6_cpu_preview)
assert np.isclose(q6_mx_cpu_val, q6_pd, rtol=1e-2, atol=1e-3), "Q6 MXFrame CPU vs Pandas mismatch"

if POLARS_AVAILABLE:
    _assert_q1_close(q1_cpu_preview, q1_pl.to_pandas(), "Q1 MXFrame CPU vs Polars")
    assert np.isclose(q6_mx_cpu_val, q6_pl, rtol=1e-2, atol=1e-3), "Q6 MXFrame CPU vs Polars mismatch"

print("Correctness checks passed for MXFrame CPU against selected baselines.")

Correctness checks passed for MXFrame CPU against selected baselines.


## 5) Timing helpers and GPU readiness

In [ ]:
RUNS = 5
WARMUP = 2
CACHE_MISS_RUNS = 3
CACHE_HIT_RUNS = 5
SWEEP_SIZES = [1_000_000, 2_000_000, 5_000_000]
SWEEP_RUNS = 3
SWEEP_WARMUP = 1

def _time_runs(fn, runs: int = RUNS, warmup: int = WARMUP):
    for _ in range(warmup):
        fn()
    samples = []
    last = None
    for _ in range(runs):
        t0 = time.perf_counter()
        last = fn()
        samples.append((time.perf_counter() - t0) * 1000.0)
    return samples, last

def _stats(samples):
    arr = np.array(samples, dtype=float)
    return {
        "min_ms": float(arr.min()),
        "avg_ms": float(arr.mean()),
        "median_ms": float(np.median(arr)),
    }

GPU_VISIBLE = _driver.accelerator_count() > 0
GPU_READY = False
GPU_REASON = "No accelerator detected"

if GPU_VISIBLE:
    try:
        _ = run_q6_mxframe(lineitem, device="gpu")
        GPU_READY = True
        GPU_REASON = "MAX-compatible GPU available"
    except Exception as e:
        GPU_READY = False
        GPU_REASON = f"GPU unavailable for MXFrame: {e}"

print(f"GPU visible : {GPU_VISIBLE}")
print(f"GPU ready   : {GPU_READY}")
print(f"GPU status  : {GPU_REASON}")
print("Benchmark modes:")
print("- CACHE-MISS: clear MXFrame caches every timed run")
print("- CACHE-HIT : reuse MXFrame caches across timed runs")
print("Runtime provenance:")
print("- Storage/interchange: PyArrow table")
print("- Kernels/graph execution: MAX graph + Mojo custom kernels")
print("- Current preprocessing path uses predicate evaluation + NumPy materialization in compile_and_run")

GPU visible : True
GPU ready   : True
GPU status  : MAX-compatible GPU available
Benchmark modes:
- CACHE-MISS: clear MXFrame caches every timed run
- CACHE-HIT : reuse MXFrame caches across timed runs
Runtime provenance:
- Storage/interchange: PyArrow table
- Kernels/graph execution: MAX graph + Mojo custom kernels
- Current preprocessing path uses PyArrow + NumPy materialization in compile_and_run


## 6) Benchmark run and console summary (Q1 + Q6)

In [ ]:
def _print_summary(title: str, rows):
    print(f"\n{title}")
    print("-" * len(title))
    print(f"{'Engine':<20} {'Min (ms)':>10} {'Avg (ms)':>10} {'Median (ms)':>12}")
    for name, stat in rows:
        print(f"{name:<20} {stat['min_ms']:>10.2f} {stat['avg_ms']:>10.2f} {stat['median_ms']:>12.2f}")


def _print_real_user_summary(title: str, rows):
    print(f"\n{title}")
    print("-" * len(title))
    print(f"{'Engine':<20} {'One-shot (ms)':>16}")
    for name, ms in rows:
        print(f"{name:<20} {ms:>16.2f}")


def _time_once(fn):
    t0 = time.perf_counter()
    out = fn()
    return (time.perf_counter() - t0) * 1000.0, out


def _print_mx_provenance(label: str):
    if not hasattr(LazyFrame, "get_last_compute_provenance"):
        print(f"[provenance:{label}] unavailable (runtime API missing)")
        return
    prov = LazyFrame.get_last_compute_provenance() or {}
    ops_used = prov.get("custom_ops_used", {})
    kernels = ", ".join(f"{k}:{v}" for k, v in sorted(ops_used.items())) if ops_used else "none"
    print(
        f"[provenance:{label}] device={prov.get('device')} rows={prov.get('rows')} "
        f"grouped={prov.get('grouped')} n_groups={prov.get('n_groups')} cache_hit={prov.get('cache_hit')}"
    )
    print(
        f"[provenance:{label}] custom_ops={kernels} "
        f"pyarrow_pre={prov.get('uses_pyarrow_preprocessing')} "
        f"pyarrow_row_filter={prov.get('uses_pyarrow_row_filtering')} "
        f"masked_pushdown={prov.get('uses_masked_filter_pushdown')} "
        f"numpy_mat={prov.get('uses_numpy_materialization')}"
    )


def _clear_compiler_cache():
    if hasattr(LazyFrame, "clear_compiler_cache"):
        LazyFrame.clear_compiler_cache()


def _mx_cache_miss_stats(fn, runs: int = CACHE_MISS_RUNS):
    samples = []
    last = None
    for _ in range(runs):
        _clear_compiler_cache()
        t0 = time.perf_counter()
        last = fn()
        samples.append((time.perf_counter() - t0) * 1000.0)
    return _stats(samples), last


def _mx_cache_hit_stats(fn, runs: int = CACHE_HIT_RUNS, warmup: int = WARMUP):
    _clear_compiler_cache()
    fn()
    samples, last = _time_runs(fn, runs=runs, warmup=warmup)
    return _stats(samples), last


def _run_q1_pandas_df(pdf: pd.DataFrame) -> pd.DataFrame:
    q1 = pdf[pdf["l_shipdate"] <= CUTOFF_Q1].copy()
    q1["disc_price"] = q1["l_extendedprice"] * (1.0 - q1["l_discount"])
    q1["charge"] = q1["disc_price"] * (1.0 + q1["l_tax"])
    return (
        q1.groupby(["l_returnflag", "l_linestatus"], as_index=False)
        .agg(
            sum_qty=("l_quantity", "sum"),
            sum_base_price=("l_extendedprice", "sum"),
            sum_disc_price=("disc_price", "sum"),
            sum_charge=("charge", "sum"),
            avg_qty=("l_quantity", "mean"),
            avg_price=("l_extendedprice", "mean"),
            avg_disc=("l_discount", "mean"),
            count_order=("l_quantity", "count"),
        )
        .sort_values(["l_returnflag", "l_linestatus"])
        .reset_index(drop=True)
    )


def _run_q6_pandas_df(pdf: pd.DataFrame) -> float:
    mask = (
        (pdf["l_shipdate"] >= Q6_DATE_LO)
        & (pdf["l_shipdate"] < Q6_DATE_HI)
        & (pdf["l_discount"] >= Q6_DISC_LO)
        & (pdf["l_discount"] <= Q6_DISC_HI)
        & (pdf["l_quantity"] < Q6_QTY_HI)
    )
    return float((pdf.loc[mask, "l_extendedprice"] * pdf.loc[mask, "l_discount"]).sum())


def _run_q1_polars_df(pl_df):
    return (
        pl_df
        .filter(pl.col("l_shipdate") <= CUTOFF_Q1)
        .with_columns([
            (pl.col("l_extendedprice") * (1.0 - pl.col("l_discount"))).alias("disc_price"),
            (pl.col("l_extendedprice") * (1.0 - pl.col("l_discount")) * (1.0 + pl.col("l_tax"))).alias("charge"),
        ])
        .group_by(["l_returnflag", "l_linestatus"])
        .agg([
            pl.col("l_quantity").sum().alias("sum_qty"),
            pl.col("l_extendedprice").sum().alias("sum_base_price"),
            pl.col("disc_price").sum().alias("sum_disc_price"),
            pl.col("charge").sum().alias("sum_charge"),
            pl.col("l_quantity").mean().alias("avg_qty"),
            pl.col("l_extendedprice").mean().alias("avg_price"),
            pl.col("l_discount").mean().alias("avg_disc"),
            pl.col("l_quantity").count().alias("count_order"),
        ])
        .sort(["l_returnflag", "l_linestatus"])
    )


def _run_q6_polars_df(pl_df) -> float:
    out = (
        pl_df
        .filter(
            (pl.col("l_shipdate") >= Q6_DATE_LO)
            & (pl.col("l_shipdate") < Q6_DATE_HI)
            & (pl.col("l_discount") >= Q6_DISC_LO)
            & (pl.col("l_discount") <= Q6_DISC_HI)
            & (pl.col("l_quantity") < Q6_QTY_HI)
        )
        .select((pl.col("l_extendedprice") * pl.col("l_discount")).sum().alias("revenue"))
    )
    return float(out["revenue"][0])


print("\n=== Q1 Real-User One-Shot (same Arrow input) ===")
q1_real_rows = []

_clear_compiler_cache()
q1_mx_cpu_real_ms, q1_mx_cpu_real_last = _time_once(lambda: run_q1_mxframe(lineitem, device="cpu"))
q1_real_rows.append(("MXFrame CPU", q1_mx_cpu_real_ms))
_print_mx_provenance("Q1 CPU real-user")

if GPU_READY:
    _clear_compiler_cache()
    q1_mx_gpu_real_ms, q1_mx_gpu_real_last = _time_once(lambda: run_q1_mxframe(lineitem, device="gpu"))
    q1_real_rows.append(("MXFrame GPU", q1_mx_gpu_real_ms))
    _print_mx_provenance("Q1 GPU real-user")
    _assert_q1_close(q1_mx_gpu_real_last, q1_mx_cpu_real_last.to_pandas(), "Q1 MXFrame GPU vs MXFrame CPU (real-user)")

q1_pd_real_ms, _ = _time_once(lambda: run_q1_pandas(lineitem))
q1_real_rows.append(("Pandas (from Arrow)", q1_pd_real_ms))

if POLARS_AVAILABLE:
    q1_pl_real_ms, _ = _time_once(lambda: run_q1_polars(lineitem))
    q1_real_rows.append(("Polars (from Arrow)", q1_pl_real_ms))

_print_real_user_summary(f"Q1 Real-User One-Shot ({N:,} rows)", q1_real_rows)


print("\n=== Q1 Apples-to-Apples ===")

q1_rows_miss = []
q1_rows_hit = []

q1_cpu_miss_stat, q1_cpu_miss_last = _mx_cache_miss_stats(
    lambda: run_q1_mxframe(lineitem, device="cpu"),
    runs=CACHE_MISS_RUNS,
)
q1_rows_miss.append(("MXFrame CPU", q1_cpu_miss_stat))
_print_mx_provenance("Q1 CPU miss")

q1_cpu_hit_stat, q1_cpu_hit_last = _mx_cache_hit_stats(
    lambda: run_q1_mxframe(lineitem, device="cpu"),
    runs=CACHE_HIT_RUNS,
    warmup=WARMUP,
)
q1_rows_hit.append(("MXFrame CPU", q1_cpu_hit_stat))
_print_mx_provenance("Q1 CPU hit")

if GPU_READY:
    q1_gpu_miss_stat, q1_gpu_miss_last = _mx_cache_miss_stats(
        lambda: run_q1_mxframe(lineitem, device="gpu"),
        runs=CACHE_MISS_RUNS,
    )
    q1_rows_miss.append(("MXFrame GPU", q1_gpu_miss_stat))
    _print_mx_provenance("Q1 GPU miss")

    q1_gpu_hit_stat, q1_gpu_hit_last = _mx_cache_hit_stats(
        lambda: run_q1_mxframe(lineitem, device="gpu"),
        runs=CACHE_HIT_RUNS,
        warmup=WARMUP,
    )
    q1_rows_hit.append(("MXFrame GPU", q1_gpu_hit_stat))
    _print_mx_provenance("Q1 GPU hit")

    _assert_q1_close(q1_gpu_hit_last, q1_cpu_hit_last.to_pandas(), "Q1 MXFrame GPU vs MXFrame CPU")

q1_pd_miss_samples, _ = _time_runs(lambda: run_q1_pandas(lineitem), runs=CACHE_MISS_RUNS, warmup=1)
q1_rows_miss.append(("Pandas (from Arrow)", _stats(q1_pd_miss_samples)))

q1_pdf = lineitem.to_pandas()
q1_pd_hit_samples, _ = _time_runs(lambda: _run_q1_pandas_df(q1_pdf), runs=CACHE_HIT_RUNS, warmup=WARMUP)
q1_rows_hit.append(("Pandas (preloaded)", _stats(q1_pd_hit_samples)))

if POLARS_AVAILABLE:
    q1_pl_miss_samples, _ = _time_runs(lambda: run_q1_polars(lineitem), runs=CACHE_MISS_RUNS, warmup=1)
    q1_rows_miss.append(("Polars (from Arrow)", _stats(q1_pl_miss_samples)))

    q1_pl_df = pl.from_arrow(lineitem)
    q1_pl_hit_samples, _ = _time_runs(lambda: _run_q1_polars_df(q1_pl_df), runs=CACHE_HIT_RUNS, warmup=WARMUP)
    q1_rows_hit.append(("Polars (preloaded)", _stats(q1_pl_hit_samples)))

_print_summary(f"Q1 Cache-MISS ({N:,} rows)", q1_rows_miss)
_print_summary(f"Q1 Cache-HIT ({N:,} rows)", q1_rows_hit)


print("\n=== Q6 Apples-to-Apples ===")

q6_rows_miss = []
q6_rows_hit = []

q6_cpu_miss_stat, q6_cpu_miss_last = _mx_cache_miss_stats(
    lambda: run_q6_mxframe(lineitem, device="cpu"),
    runs=CACHE_MISS_RUNS,
)
q6_rows_miss.append(("MXFrame CPU", q6_cpu_miss_stat))
_print_mx_provenance("Q6 CPU miss")

q6_cpu_hit_stat, q6_cpu_hit_last = _mx_cache_hit_stats(
    lambda: run_q6_mxframe(lineitem, device="cpu"),
    runs=CACHE_HIT_RUNS,
    warmup=WARMUP,
)
q6_rows_hit.append(("MXFrame CPU", q6_cpu_hit_stat))
q6_cpu_hit_val = _extract_q6_revenue(q6_cpu_hit_last)
_print_mx_provenance("Q6 CPU hit")

if GPU_READY:
    q6_gpu_miss_stat, q6_gpu_miss_last = _mx_cache_miss_stats(
        lambda: run_q6_mxframe(lineitem, device="gpu"),
        runs=CACHE_MISS_RUNS,
    )
    q6_rows_miss.append(("MXFrame GPU", q6_gpu_miss_stat))
    _print_mx_provenance("Q6 GPU miss")

    q6_gpu_hit_stat, q6_gpu_hit_last = _mx_cache_hit_stats(
        lambda: run_q6_mxframe(lineitem, device="gpu"),
        runs=CACHE_HIT_RUNS,
        warmup=WARMUP,
    )
    q6_rows_hit.append(("MXFrame GPU", q6_gpu_hit_stat))

    q6_gpu_hit_val = _extract_q6_revenue(q6_gpu_hit_last)
    assert np.isclose(q6_gpu_hit_val, q6_cpu_hit_val, rtol=1e-2, atol=1e-3), "Q6 GPU vs CPU mismatch"
    _print_mx_provenance("Q6 GPU hit")

q6_pd_miss_samples, _ = _time_runs(lambda: run_q6_pandas(lineitem), runs=CACHE_MISS_RUNS, warmup=1)
q6_rows_miss.append(("Pandas (from Arrow)", _stats(q6_pd_miss_samples)))

q6_pdf = lineitem.to_pandas()
q6_pd_hit_samples, _ = _time_runs(lambda: _run_q6_pandas_df(q6_pdf), runs=CACHE_HIT_RUNS, warmup=WARMUP)
q6_rows_hit.append(("Pandas (preloaded)", _stats(q6_pd_hit_samples)))

if POLARS_AVAILABLE:
    q6_pl_miss_samples, _ = _time_runs(lambda: run_q6_polars(lineitem), runs=CACHE_MISS_RUNS, warmup=1)
    q6_rows_miss.append(("Polars (from Arrow)", _stats(q6_pl_miss_samples)))

    q6_pl_df = pl.from_arrow(lineitem)
    q6_pl_hit_samples, _ = _time_runs(lambda: _run_q6_polars_df(q6_pl_df), runs=CACHE_HIT_RUNS, warmup=WARMUP)
    q6_rows_hit.append(("Polars (preloaded)", _stats(q6_pl_hit_samples)))

_print_summary(f"Q6 Cache-MISS ({N:,} rows)", q6_rows_miss)
_print_summary(f"Q6 Cache-HIT ({N:,} rows)", q6_rows_hit)
print(f"Q6 revenue (MXFrame CPU hit): {q6_cpu_hit_val:.4f}")


if GPU_READY:
    print("\nGPU scaling sweep (MXFrame only, CACHE-HIT oriented)")
    print("----------------------------------------------------")
    print(f"{'Rows':>10} {'Q1 CPU':>10} {'Q1 GPU':>10} {'Q1 Speedup':>12} {'Q6 CPU':>10} {'Q6 GPU':>10} {'Q6 Speedup':>12}")

    for sweep_n in SWEEP_SIZES:
        sweep_tbl = lineitem if sweep_n == N else make_lineitem_arrow(sweep_n, seed=42)

        q1_cpu_samples_s, q1_cpu_last_s = _time_runs(
            lambda: run_q1_mxframe(sweep_tbl, device="cpu"),
            runs=SWEEP_RUNS,
            warmup=SWEEP_WARMUP,
        )
        q1_gpu_samples_s, q1_gpu_last_s = _time_runs(
            lambda: run_q1_mxframe(sweep_tbl, device="gpu"),
            runs=SWEEP_RUNS,
            warmup=SWEEP_WARMUP,
        )
        q1_cpu_min_s = _stats(q1_cpu_samples_s)["min_ms"]
        q1_gpu_min_s = _stats(q1_gpu_samples_s)["min_ms"]
        q1_speedup_s = q1_cpu_min_s / q1_gpu_min_s

        q6_cpu_samples_s, q6_cpu_last_s = _time_runs(
            lambda: run_q6_mxframe(sweep_tbl, device="cpu"),
            runs=SWEEP_RUNS,
            warmup=SWEEP_WARMUP,
        )
        q6_gpu_samples_s, q6_gpu_last_s = _time_runs(
            lambda: run_q6_mxframe(sweep_tbl, device="gpu"),
            runs=SWEEP_RUNS,
            warmup=SWEEP_WARMUP,
        )
        q6_cpu_min_s = _stats(q6_cpu_samples_s)["min_ms"]
        q6_gpu_min_s = _stats(q6_gpu_samples_s)["min_ms"]
        q6_speedup_s = q6_cpu_min_s / q6_gpu_min_s

        q6_cpu_val_s = _extract_q6_revenue(q6_cpu_last_s)
        q6_gpu_val_s = _extract_q6_revenue(q6_gpu_last_s)
        assert np.isclose(q6_gpu_val_s, q6_cpu_val_s, rtol=1e-2, atol=1e-3), f"Q6 GPU vs CPU mismatch at N={sweep_n:,}"

        _assert_q1_close(
            q1_gpu_last_s,
            q1_cpu_last_s.to_pandas(),
            f"Q1 GPU vs CPU mismatch at N={sweep_n:,}",
        )

        print(
            f"{sweep_n:>10,} "
            f"{q1_cpu_min_s:>10.2f} {q1_gpu_min_s:>10.2f} {q1_speedup_s:>12.2f} "
            f"{q6_cpu_min_s:>10.2f} {q6_gpu_min_s:>10.2f} {q6_speedup_s:>12.2f}"
        )


=== Q1 Real-User One-Shot (same Arrow input) ===
[provenance:Q1 CPU real-user] device=cpu rows=1000000 grouped=True n_groups=6 cache_hit=False
[provenance:Q1 CPU real-user] custom_ops=group_sum:11 pyarrow_pre=True pyarrow_row_filter=False masked_pushdown=True numpy_mat=True
[provenance:Q1 GPU real-user] device=gpu rows=1000000 grouped=True n_groups=6 cache_hit=False
[provenance:Q1 GPU real-user] custom_ops=group_sum:11 pyarrow_pre=True pyarrow_row_filter=False masked_pushdown=True numpy_mat=True

Q1 Real-User One-Shot (1,000,000 rows)
--------------------------------------
Engine                  One-shot (ms)
MXFrame CPU                   4187.04
MXFrame GPU                   3985.58
Pandas (from Arrow)            332.06
Polars (from Arrow)             78.00

=== Q1 Apples-to-Apples ===
[provenance:Q1 CPU miss] device=cpu rows=1000000 grouped=True n_groups=6 cache_hit=False
[provenance:Q1 CPU miss] custom_ops=group_sum:11 pyarrow_pre=True pyarrow_row_filter=False masked_pushdown=True